# Session 4 — Your turn: Three transformer layouts

The three layouts differ in who may read whom, and in what the model is trained to recover. Write both as small functions.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: Three masks

Write `mask_for(layout, n_source, n_target=0)` returning a boolean matrix where True means "may read": for `"encoder"` an `n_source x n_source` matrix of all True; for `"decoder"` an `n_source x n_source` lower-triangular matrix;
for `"cross"` an `n_target x n_source` matrix of all True (every target token reads the whole source). Then fill `allowed_pairs` with the number of True entries for an encoder, a decoder and a cross mask over 6 source tokens (and 4 target tokens for cross).

In [2]:
def mask_for(layout, n_source, n_target=0):
    if layout == "encoder": return np.ones((n_source, n_source), dtype=bool)
    if layout == "decoder": return np.tril(np.ones((n_source, n_source), dtype=bool))
    if layout == "cross": return np.ones((n_target, n_source), dtype=bool)
    raise ValueError(layout)


allowed_pairs = {"encoder": int(mask_for("encoder", 6).sum()), "decoder": int(mask_for("decoder", 6).sum()), "cross": int(mask_for("cross", 6, 4).sum())}
print(allowed_pairs)

{'encoder': 36, 'decoder': 21, 'cross': 24}


In [3]:
check("the encoder lets every token read every token (36 pairs)", lambda: allowed_pairs["encoder"] == 36)
check("the decoder lets each token read itself and earlier ones (21 pairs)", lambda: allowed_pairs["decoder"] == 21)
check("cross-attention is target x source (24 pairs)", lambda: allowed_pairs["cross"] == 24)
check("a decoder mask has nothing above the diagonal", lambda: not np.triu(mask_for("decoder", 5), 1).any())
summary()

PASS  the encoder lets every token read every token (36 pairs)
PASS  the decoder lets each token read itself and earlier ones (21 pairs)
PASS  cross-attention is target x source (24 pairs)
PASS  a decoder mask has nothing above the diagonal

4 of 4 checks passed.  Well done.


**Why it works.** The three layouts share the same block and differ only in this matrix. n(n+1)/2 allowed pairs is why a decoder can be trained on every position of a text in one pass: each position's prediction sees exactly its own past.

## Task 2: BERT's 80 / 10 / 10 corruption

Write `corrupt(ids, rng, vocab_size=1000, mask_id=0)`: choose each position with probability 0.15; of the chosen positions, replace 80% by `mask_id`, 10% by a random id from `1` to `vocab_size - 1`, and leave 10% unchanged. Return `(corrupted, chosen)`, where `chosen` is a boolean array marking the positions the loss is computed on.

In [4]:
def corrupt(ids, rng, vocab_size=1000, mask_id=0):
    ids = np.asarray(ids)
    chosen = rng.random(len(ids)) < 0.15
    u = rng.random(len(ids))
    corrupted = ids.copy()
    corrupted[chosen & (u < 0.8)] = mask_id
    randomise = chosen & (u >= 0.8) & (u < 0.9)
    corrupted[randomise] = rng.integers(1, vocab_size, randomise.sum())
    return corrupted, chosen


ids = np.arange(1, 20001) % 999 + 1
corrupted, chosen = corrupt(ids, np.random.default_rng(0))
print(chosen.mean(), (corrupted[chosen] == 0).mean())

0.14895 0.7901980530379322


In [5]:
check("about 15% of positions are chosen", lambda: abs(chosen.mean() - 0.15) < 0.01)
check("about 80% of the chosen positions become the mask", lambda: abs((corrupted[chosen] == 0).mean() - 0.8) < 0.02)
check("positions that were not chosen are untouched", lambda: np.array_equal(corrupted[~chosen], ids[~chosen]))
check("about 10% of the chosen positions are left unchanged or re-drawn to the same id", lambda: 0.08 < (corrupted[chosen] == ids[chosen]).mean() < 0.13)
summary()

PASS  about 15% of positions are chosen
PASS  about 80% of the chosen positions become the mask
PASS  positions that were not chosen are untouched
PASS  about 10% of the chosen positions are left unchanged or re-drawn to the same id

4 of 4 checks passed.  Well done.


**Why it works.** If every chosen token were replaced by [MASK], the model would only ever learn to fill masks; the random and unchanged cases force it to check every token, and the session measures how differently BERT handles the three treatments (70% recovered for masks, 99% for unchanged ones).